# Project Part 3: Base Map

## Overview

In Part 2 you produced a cleaned, sentiment-annotated dataset and a written analysis. Now you will build the **base map** that will power the interactive visualization on your team website.

This map is the foundation for the flythrough you will build in **Part 4**. It needs to:

- Display all geoparsed locations for your school, sized by the number of times it occurs and colored by sentiment
- Be clean enough to serve as a standalone visualization
- Export in a format usable by the flythrough template

---

## ⚠️ Before You Begin

You must have completed **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)** and created a cleaned data set of your school's data. You should see a copy of it in your school's data folder called `{SCHOOL}_geoparsed_long_cleaned_sentiment.csv`, where `{SCHOOL}` is your school's abbreviation. If along the way you find that the data was not properly cleaned, you have to repeat steps 2–5 in **[project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb)**.

In [4]:
# ============================================================
# STEP 0: Set your school (must match Part 2)
# ============================================================

SCHOOL = "UNC"   # <-- change this to your school

import pandas as pd
import plotly.express as px
import json

---

## 📖 1 Follow Along — Load and Aggregate Your Processed Data

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The cell below loads sentiment data for **both your school and JMU**, then aggregates each dataset to one row per unique place. The two datasets are combined into a single DataFrame before any classification. This mirrors the approach from [Part 2](project_part_2_whitepaper.ipynb) and ensures that the Jenks size and color bins computed in Section 2 are shared across both schools — so equivalent bubbles carry equivalent meaning on the map.

The `school` column records which school each place belongs to; it will appear in the hover tooltip.

If your school's file is not found, complete `project_part_1_data_pipeline.ipynb` for your school first.

In [5]:
import os
import numpy as np

SCHOOL_DATA_PATH = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long_cleaned_sentiment.pickle'
JMU_PICKLE_PATH  = '../data/JMU/JMU_geoparsed_long_cleaned_sentiment.pickle'
JMU_CSV_PATH     = '../data/JMU/JMU_geoparsed_long_cleaned_sentiment.csv'
JMU_BACKUP_PATH  = '../data/JMU/JMU_geoparsed_long_backup_sentiment.csv'

df_places = None

def _aggregate(df_raw, school_name):
    """Aggregate a per-sentence DataFrame to one row per place."""
    return (
        df_raw
        .dropna(subset=['place', 'latitude', 'longitude'])
        .astype({'latitude': float, 'longitude': float})
        .groupby('place', sort=False)
        .agg(
            location_count=('place', 'size'),
            latitude=('latitude', 'first'),
            longitude=('longitude', 'first'),
            sentences=('sentences', lambda x: ' | '.join(str(s) for s in list(x)[:5])),
            avg_roberta_compound=('roberta_compound', 'mean'),
            place_type=('place_type', 'first'),
        )
        .reset_index()
        .assign(school=school_name)
    )

# ── Load school data ──────────────────────────────────────────────────────────
if not os.path.exists(SCHOOL_DATA_PATH):
    print(f'⛔ File not found: {SCHOOL_DATA_PATH}')
    print('   Complete project_part_1_data_pipeline.ipynb for your school first.')
else:
    df_school_raw = pd.read_pickle(SCHOOL_DATA_PATH)
    print(f'✅ {SCHOOL}: loaded {len(df_school_raw):,} rows')

    # ── Load JMU data (pickle → primary CSV → backup CSV) ────────────────────────
    if os.path.exists(JMU_PICKLE_PATH):
        df_jmu_raw = pd.read_pickle(JMU_PICKLE_PATH)
    elif os.path.exists(JMU_CSV_PATH):
        df_jmu_raw = pd.read_csv(JMU_CSV_PATH)
        print('⚠️  Using JMU primary CSV (pickle not found).')
    elif os.path.exists(JMU_BACKUP_PATH):
        df_jmu_raw = pd.read_csv(JMU_BACKUP_PATH)
        print('⚠️  Using JMU backup sentiment data.')
    else:
        raise FileNotFoundError('No JMU sentiment data found in ../data/JMU/.')
    print(f'✅ JMU:  loaded {len(df_jmu_raw):,} rows')

    # ── Aggregate each school to one row per place ────────────────────────────────
    df_school = _aggregate(df_school_raw, SCHOOL)
    df_jmu    = _aggregate(df_jmu_raw,    'JMU')

    # ── Combine — Jenks bins in Section 2 will be shared across both schools ──────
    df_places = pd.concat([df_jmu, df_school], ignore_index=True)

    print(f'\n✅ JMU: {len(df_jmu):,} unique places  |  {SCHOOL}: {len(df_school):,} unique places')
    print(f'✅ Combined: {len(df_places):,} total place records')
    print(f'\nTop place types (combined):')
    print(df_places['place_type'].value_counts(dropna=False).head(8).to_string())
    print(f'\nSentiment range: {df_places["avg_roberta_compound"].min():.3f} to {df_places["avg_roberta_compound"].max():.3f}')
    df_places.head(3)

✅ UNC: loaded 1,263 rows
⚠️  Using JMU backup sentiment data.
✅ JMU:  loaded 876 rows

✅ JMU: 307 unique places  |  UNC: 369 unique places
✅ Combined: 676 total place records

Top place types (combined):
place_type
City               204
Building           156
State              103
Country             56
Natural Feature     41
None                40
Neighborhood        28
Region              20

Sentiment range: -0.949 to 0.984


---

## 2 Build Your Map

Fill in the design brief table below **before** changing any code. Every parameter should be a deliberate choice — not an accepted default. Reference your observations from [Lesson 6](../lesson_6_mapping_fundamentals/lesson_6_mapping_fundamentals.ipynb) Decisions 1–7.

> ✍️ **Activity:** Complete every row in the design brief, then set each matching variable in the code cell and run it to generate your map.

### Design Brief

**Fill in every row before touching the code cell.** Reference Decisions 1–7 from Lesson 6.

| Design Decision | Variable | Your Choice | Reasoning |
|---|---|---|---|
| Filtering threshold | `MIN_COUNT` | | Which places are worth showing? |
| Place type filter | `PLACE_TYPES` | | Which geographic scales belong in your story? |
| Size classification | `N_SIZE_CLASSES` | | How many Jenks size classes? (3–5) |
| Color buckets | `N_COLOR_BUCKETS` | | How many sentiment color classes? (3–7) |
| Color scale | `COLOR_SCALE` | | Which scale is honest and accessible? |
| Maximum bubble size | `SIZE_MAX` | | How large should the biggest bubble be in pixels? |
| Base map style | `MAP_STYLE` | | What tone does the background set? |
| Center coordinates | `CENTER` | | Reference viewport — flythrough will override |
| Zoom level | `ZOOM` | | Reference zoom — flythrough will override |

> 👉 **Note:** *Your submitted map must differ from the default values in at least three deliberate ways, each justifiable from this brief.*


In [ ]:

import mapclassify
import plotly.colors as pc

if df_places is None:
    print('⛔ No data — run the cells in Section 1 first.')
else:
    # ── Your design decisions — fill in every value, then run ────────────────────
    MIN_COUNT       = 3              # ← Decision 1: minimum post count per location
    PLACE_TYPES     = None           # ← Decision 2: list e.g. ['City', 'Building'], or None for all types
    N_SIZE_CLASSES  = 4              # ← Decision 3: number of Jenks size classes (try 3–5)
    N_COLOR_BUCKETS = 5              # ← Decision 4: number of sentiment color buckets (try 3–7)
    COLOR_SCALE     = 'RdYlGn'      # ← Decision 5: 'RdYlGn', 'RdBu', 'Spectral', 'Viridis'
    SIZE_MAX        = 18             # ← maximum bubble diameter in pixels (try 12–30)
    MAP_STYLE       = 'carto-positron'  # ← Decision 6: 'carto-positron', 'carto-darkmatter', 'open-street-map'
    CENTER          = {"lat": 37.5, "lon": -78.0}  # ← reference only; flythrough will override this
    ZOOM            = 6              # ← reference only; flythrough will override this

    # ── Filter ────────────────────────────────────────────────────────────────────
    if PLACE_TYPES is not None:
        df_work = df_places[
            (df_places['location_count'] >= MIN_COUNT) &
            df_places['place_type'].isin(PLACE_TYPES)
        ].copy()
    else:
        df_work = df_places[df_places['location_count'] >= MIN_COUNT].copy()

    # ── Assign a stable, memorable ID to every location ──────────────────────────
    # IDs are school abbreviation + row number within that school, e.g. JMU1, UNC34.
    # Hover over any bubble to see its ID — you will use these IDs in Section 3
    # to tell the flythrough which locations to visit.
    df_work = df_work.reset_index(drop=True)
    df_work['id'] = df_work['school'] + (df_work.groupby('school').cumcount() + 1).astype(str)

    _n_jmu = (df_work['school'] == 'JMU').sum()
    _n_sch = (df_work['school'] == SCHOOL).sum()
    print(f'── After filtering: JMU {_n_jmu} locations (JMU1–JMU{_n_jmu}), {SCHOOL} {_n_sch} locations ({SCHOOL}1–{SCHOOL}{_n_sch}) ──')

    # ── Size classification — Jenks on combined data ──────────────────────────────
    _jnb_s = mapclassify.NaturalBreaks(df_work['location_count'].values, k=N_SIZE_CLASSES)
    df_work['size_class'] = (_jnb_s.yb + 1).astype(float)

    # ── Color classification — Jenks on combined data ─────────────────────────────
    scores  = df_work['avg_roberta_compound']
    _jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
    _breaks = _jnb_c.bins
    _lo     = scores.min()
    _labels = []
    for _hi in _breaks:
        _labels.append(f"{_lo:.2f} to {_hi:.2f}")
        _lo = _hi
    df_work['color_class'] = pd.cut(scores, bins=[-float('inf')] + list(_breaks), labels=_labels)
    _palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
    _color_map = dict(zip(_labels, _palette))

    # ── Build the map ─────────────────────────────────────────────────────────────
    fig = px.scatter_map(
        df_work,
        lat='latitude', lon='longitude',
        size='size_class',
        color='color_class',
        hover_name='place',
        hover_data={
            'id': True,
            'school': True,
            'avg_roberta_compound': ':.3f',
            'location_count': True,
            'place_type': True,
            'size_class': False,
            'color_class': False,
            'latitude': False,
            'longitude': False,
        },
        color_discrete_map=_color_map,
        category_orders={'color_class': _labels},
        size_max=SIZE_MAX,
        map_style=MAP_STYLE,
        center=CENTER,
        zoom=ZOOM,
        height=650,
        title=f'JMU & {SCHOOL} — Base Map  ({len(df_work):,} locations)',
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
    fig.show()


── After filtering: JMU 55, UNC 73  (total 128) ──
── IDs run from 0 to 127 ──



---

## ✍️ 3 Plan Your Flythrough Journey

The flythrough is your team's guided tour through the data. Rather than automatically selecting locations by score or count, you choose the stops because you have a story to tell.

### Step 1 — Explore the map above

Hover over locations that interest you. The hover tooltip shows each location's **ID** — a number you will use below to reference it. Note down:

- The ID
- The place name (exactly as it appears in the hover)
- Why you find it interesting — sentiment, count, location, connection to your hypothesis

Aim for **5–10 stops** that form a coherent narrative. Think spatially: a good flythrough has a sense of movement (e.g. local → regional → national, or JMU → your school → comparison).

### Step 2 — Gather your images

For each stop you plan to include, you need **one image**. Upload your images to the `project_mapping_emotions/images/` folder in your repository. Name them clearly (e.g. `dhall.jpg`, `franklin_street.jpg`). You can use photos you took, Creative Commons images, or screenshots from Google Maps Street View.

### Step 3 — Fill in `FLYTHROUGH_LOCATIONS` below

Each entry in the list is one stop on your tour. Fill in all four fields:

| Field | What to enter |
|---|---|
| `id` | The number from the hover tooltip |
| `name` | The place name exactly as shown in the hover |
| `description` | 2–3 sentences describing the place and what the data reveals about it |
| `media` | Relative path to your image, e.g. `'images/dhall.jpg'` |

> 👉 **Note:** *The code cell below validates your entries and generates the JSON for `flythrough_config.js`. If an ID is not found, it will tell you which one failed.*


In [ ]:

# ── Flythrough framing — text and style for the opening/closing cards and sidebar ──
FLYTHROUGH_CONFIG = {
    'title':    'Our Flythrough Title',                                        # ← shown at the start of the flythrough
    'subtitle': 'A one-line thesis statement.',                                # ← shown beneath the title
    'theme':    'light',                                                        # ← sidebar color theme: 'light' or 'dark'
    'intro':    'Write 1–2 sentences of opening context for your audience.',   # ← opening card text
    'outro':    'Write 1–2 sentences reflecting on what the tour revealed.',   # ← closing card text
}

# ── Your flythrough stops — fill in each entry, then run the next cell ────
# Hover over a bubble in the map above to find its id (e.g. 'JMU1', 'UNC34').
# Upload images to project_mapping_emotions/images/ before running the export.

FLYTHROUGH_LOCATIONS = [
    {
        'chapter':     'Introduction',          # ← your label for this stop (e.g. 'JMU 1', 'COVID Era')
        'id':          'JMU1',                  # ← id from hover tooltip, e.g. 'JMU1', 'UNC12'
        'name':        'Place Name Here',       # ← name as shown in hover
        'description': 'Write 2–3 sentences describing this place and what the data reveals about it.',
        'media':       'images/your_image.jpg', # ← upload image to project_mapping_emotions/images/
        # ── Optional fields — delete the leading # to enable ──────────────────────
        # 'date_range':  (2018, 2019),          # show data from these years in this chapter
        # 'show_school': 'both',                # which school to highlight: 'JMU', your school, or 'both'
        # 'quote':       'A representative quote from the data, displayed as an annotation at this location.',
        # 'transition':  'normal',              # camera travel speed: 'slow', 'normal' (default), 'fast'
        # 'zoom':        15,                    # camera zoom level (default: 15)
        # 'center':      {'latitude': 0.0, 'longitude': 0.0},  # custom camera center (default: location coordinates)
        # 'opacity':     1.0,                   # this marker's opacity 0.0–1.0 (dim others to make it pop)
    },
    {
        'chapter':     'JMU 1',
        'id':          'JMU2',
        'name':        'Place Name Here',
        'description': 'Write 2–3 sentences describing this place and what the data reveals about it.',
        'media':       'images/your_image.jpg',
        # 'date_range':  (2020, 2022),
        # 'show_school': 'JMU',
        # 'quote':       'A representative quote from the data, displayed as an annotation at this location.',
        # 'transition':  'normal',
        # 'zoom':        15,
        # 'center':      {'latitude': 0.0, 'longitude': 0.0},
        # 'opacity':     1.0,
    },
    {
        'chapter':     'JMU 2',
        'id':          'JMU3',
        'name':        'Place Name Here',
        'description': 'Write 2–3 sentences describing this place and what the data reveals about it.',
        'media':       'images/your_image.jpg',
        # 'date_range':  (2020, 2022),
        # 'show_school': 'JMU',
        # 'quote':       'A representative quote from the data, displayed as an annotation at this location.',
        # 'transition':  'normal',
        # 'zoom':        15,
        # 'center':      {'latitude': 0.0, 'longitude': 0.0},
        # 'opacity':     1.0,
    },
    # ── Add more stops by copying and pasting one of the blocks above ────────────
]

print(f'✅ {len(FLYTHROUGH_LOCATIONS)} stop(s) defined — run the next cell to validate and export.')


✅ 3 stop(s) defined — run the next cell to validate and export.


In [ ]:

if 'df_work' not in dir() or df_work is None:
    print('⛔ Run the map cell in Section 2 first.')
elif 'FLYTHROUGH_LOCATIONS' not in dir():
    print('⛔ Run the cell above to define FLYTHROUGH_LOCATIONS first.')
else:
    # ── Validate every entry and look up coordinates from the map data ────────────
    _id_lookup = df_work.set_index('id')
    _valid_ids = sorted(_id_lookup.index.tolist())

    chapters = []
    errors   = []

    for i, stop in enumerate(FLYTHROUGH_LOCATIONS):
        chapter = stop.get('chapter', f'Stop {i}')
        loc_id  = stop.get('id')
        if loc_id is None or loc_id not in _id_lookup.index:
            errors.append(f'  [{chapter}] id={loc_id!r} not found — check the hover tooltip for the correct id (e.g. "JMU1", "{_valid_ids[0]}")')
            continue

        row = _id_lookup.loc[loc_id]

        # ── Camera center: use custom if provided, otherwise the location's coordinates ──
        center   = stop.get('center') or {}
        cam_lat  = float(center.get('latitude',  row['latitude']))
        cam_lon  = float(center.get('longitude', row['longitude']))
        cam_zoom = stop.get('zoom', 15)

        chapter_entry = {
            'id':          f'location-{loc_id}',
            'title':       stop.get('name', row['place']),
            'description': stop.get('description', ''),
            'image':       f"./{stop.get('media', 'images/placeholder.jpg')}",
            'duration':    2000,
            'transition':  stop.get('transition', 'normal'),
            'camera': {
                'latitude':  round(cam_lat,  5),
                'longitude': round(cam_lon,  5),
                'zoom':      cam_zoom,
            },
            'location': {
                'name':         row['place'],
                'latitude':     round(float(row['latitude']),  5),
                'longitude':    round(float(row['longitude']), 5),
                'postCount':    int(row['location_count']),
                'robertaScore': round(float(row['avg_roberta_compound']), 3),
                'isJMU':        row['school'] == 'JMU',
                'opacity':      stop.get('opacity', 1.0),
            },
            'showData': 'individual',
        }

        if stop.get('quote'):
            chapter_entry['quote'] = stop['quote']

        if stop.get('date_range'):
            chapter_entry['dateRange'] = {
                'start': int(stop['date_range'][0]),
                'end':   int(stop['date_range'][1]),
            }

        if stop.get('show_school'):
            chapter_entry['showSchool'] = stop['show_school']

        chapters.append(chapter_entry)

    if errors:
        print('⛔ Fix these errors before exporting:')
        for e in errors:
            print(e)
    else:
        # ── Wrap chapters in the full flythrough config object ────────────────────
        _ft = FLYTHROUGH_CONFIG if 'FLYTHROUGH_CONFIG' in dir() else {}
        _full_config = {
            'title':    _ft.get('title',    'Mapping Emotions'),
            'subtitle': _ft.get('subtitle', ''),
            'theme':    _ft.get('theme',    'light'),
            'intro':    _ft.get('intro',    ''),
            'outro':    _ft.get('outro',    ''),
            'chapters': chapters,
        }

        print(f'✅ {len(chapters)} stop(s) validated successfully:\n')
        for stop, ch in zip(FLYTHROUGH_LOCATIONS, chapters):
            chapter    = stop.get('chapter', f'Stop {FLYTHROUGH_LOCATIONS.index(stop)}')
            cam        = ch['camera']
            opacity    = ch['location']['opacity']
            transition = ch['transition']
            quote_str  = f'  quote="{stop["quote"][:40]}…"' if stop.get('quote') else ''
            date_str   = f'  dates={stop["date_range"][0]}–{stop["date_range"][1]}' if stop.get('date_range') else ''
            school_str = f'  school={stop["show_school"]}' if stop.get('show_school') else ''
            print(f'  [{chapter}]  id={stop["id"]}  zoom={cam["zoom"]}  transition={transition}  opacity={opacity}{date_str}{school_str}  →  {ch["title"]}{quote_str}')
        print()
        print('── Paste the JSON below into flythrough_config.js, replacing the existing config object ──\n')
        print(json.dumps(_full_config, indent=2))


✅ 3 stop(s) validated successfully:

  [Introduction]  id=0  zoom=15  transition=normal  opacity=1.0  →  Place Name Here
  [JMU 1]  id=1  zoom=15  transition=normal  opacity=1.0  →  Place Name Here
  [JMU 2]  id=2  zoom=15  transition=normal  opacity=1.0  →  Place Name Here

── Paste the JSON below into flythrough_config.js, replacing the existing config object ──

{
  "title": "Our Flythrough Title",
  "subtitle": "A one-line thesis statement.",
  "theme": "light",
  "intro": "Write 1\u20132 sentences of opening context for your audience.",
  "outro": "Write 1\u20132 sentences reflecting on what the tour revealed.",
  "chapters": [
    {
      "id": "location-0",
      "title": "Place Name Here",
      "description": "Write 2\u20133 sentences describing this place and what the data reveals about it.",
      "image": "./images/your_image.jpg",
      "duration": 2000,
      "transition": "normal",
      "camera": {
        "latitude": 38.44957,
        "longitude": -78.86892,
        "z

---

## Section 4: Embed the Map in the Website

Save the final map as an HTML file so it can be linked from `index.html` or embedded in `whitepaper.html`.

**When your base map is ready, move on to `project_part_4_flythrough.ipynb`.**

In [9]:
if 'fig' not in dir():
    print('⛔ Run the map cell in Section 2 first.')
else:
    fig.write_html("base_map.html")
    print("✅ Map saved to base_map.html")

✅ Map saved to base_map.html
